[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/rotor_whitney/rotor_whitney.ipynb)

# Rotor Whitney Forms

Rotations attached to a mesh become a continuous field of orientations. Blending corner rotors traces a chord inside the unit rotor sphere; its changing orientation and changing length occupy different grades of one derivative. The example follows that local construction into a disk that wraps the sphere of directions, and frames that untwist across a curved surface.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.surfaces.rotor_whitney import render, scenarios
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, mv = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                # Scalar+
Rotor = ga.gatype.rotor()                                  # Rotor+
Point = ga.gatype.antivector()                             # Antivector-

np.set_printoptions(precision=4, suppress=True)

## Turning and stretching on a triangle

A displacement across the triangle changes both the orientation and the length of the interpolated chord. The map `2 * chord_rotors.reverse() * rotor_differential` holds both changes: its turning plane carries the edge Whitney forms, while its scalar part measures the changing chord norm. Normalization keeps the orientation and restores a rigid turn.


In [ ]:
n_triangle_subdivisions = 6
n_fine_subdivisions = 18

In [ ]:
corners, points, fine_mesh, fine_vertices, corner_rotors = scenarios.single_triangle(
    n_triangle_subdivisions, n_fine_subdivisions,
)                                                          # [corners] Point-, [samples] Point-, Mesh, Point-[fine], [corners] Rotor+
sides = corners[[2, 0, 1]] - corners[[1, 2, 0]]            # [corners] Point-
patch = (sides[0] & sides[1]) / 2                          # [] Patch-
gradients = (sides | patch.inverse()) / 2                  # [corners] Plane+
hats = 1 + (gradients & (points[:, None] - corners))       # [samples, corners] Scalar+

# Linear blending cuts inside the unit rotor sphere; normalization restores a rigid turn.
chord_rotors = (corner_rotors * hats).sum(axis=-1)         # [samples] Even+
unit_rotors = chord_rotors.normalized()                    # [samples] Rotor+
rotor_differential = (corner_rotors * (gradients & Point)).sum(axis=0)  # [] Even+ <- Point-
darboux = 2 * chord_rotors.reverse() * rotor_differential  # [samples] Even+ <- Point-

# Reversing an edge flips its transition's turning plane and preserves its scalar part.
# These match the antisymmetric Whitney forms and symmetric endpoint-product differentials.
tails, heads = [1, 2, 0], [2, 0, 1]
transitions = corner_rotors[tails].reverse() * corner_rotors[heads]  # [corners] Rotor+
edge_bivectors = 2 * transitions.select[2]                 # [corners] Bivector+
edge_sagittas = 2 * (transitions.select[0] - 1)            # [corners] Scalar+
whitney_forms = hats[:, tails] * gradients[heads] - hats[:, heads] * gradients[tails]          # [samples, corners] Plane+
symmetric_forms = hats[:, tails] * gradients[heads] + hats[:, heads] * gradients[tails]        # [samples, corners] Plane+
whitney_rates = (edge_bivectors * (whitney_forms & Point)).sum(axis=-1)  # [samples] Bivector+ <- Point-
chord_stretch = (edge_sagittas * (symmetric_forms & Point)).sum(axis=-1) # [samples] Scalar+ <- Point-
angular_velocity = whitney_rates / chord_rotors.scalar_norm_squared()    # [samples] Bivector+ <- Point-

fine_hats = 1 + (gradients & (fine_vertices.batch()[:, None] - corners))  # [fine, corners] Scalar+
fine_chords = (corner_rotors * fine_hats).sum(axis=-1)     # [fine] Even+
fine_norms_squared = fine_chords.scalar_norm_squared().field()  # Scalar+[fine]

save_figure(
    render.draw_darboux_whitney(
        corners, points, fine_mesh, fine_vertices, fine_norms_squared,
        corner_rotors, unit_rotors, whitney_rates,
    ),
    "rotor_whitney_darboux",
);

The four coloured sectors of each frame patch make its orientation visible across the triangle. The arcs turn in the bivector planes the rate map returns for the blue probe displacement. The relief gives the squared chord norm a height: unit length at the corners, dipping inside where linear interpolation cuts through the rotor sphere.

In quaternion notation, the same split reads $2\bar q\,dq=d|q|^2+\operatorname{Im}(2\bar q\,dq)$; division of the imaginary part by $|q|^2$ gives the angular-velocity form of $q/|q|$.


In [ ]:
print("Edge chord sagittas:", edge_sagittas.to_array())
print("Minimum chord norm squared:", fine_norms_squared.batch().to_array().min())


## A disk wraps the director sphere

The vertical director tilts from upward at the centre, through the horizontal plane halfway out, to downward at the rim. The disk covers the sphere of directions once: every boundary point reaches the same south pole. Its winding belongs to the projection `skyrmion_rotors >> vertical`; the complete rotor still carries the orientation around that director.


In [ ]:
n_skyrmion_divisions = 12
disk_radius = 1.0
skyrmion_twist = np.pi

In [ ]:
skyrmion_mesh, skyrmion_vertices = Mesh.triangular_disk(disk_radius, n_skyrmion_divisions)  # Mesh, Point-[vertices]
vertical = mv.z.dual()                                     # [] Point-

# The radial position and vertical direction span the plane of turning.
# At a half turn across the disk, its director runs from the north to the south pole.
skyrmion_rotors = ((-skyrmion_twist / 2) * skyrmion_vertices.commutator(vertical)).exp()  # Rotor+[vertices]
directors = skyrmion_rotors >> vertical                    # Point-[vertices]
scalar_parts = skyrmion_rotors.select[0]                   # Scalar+[vertices]
bivector_norms = skyrmion_rotors.select[2].norm()          # Scalar+[vertices]

skyrmion_edges = skyrmion_mesh.d[0] * skyrmion_vertices    # Point-[edges]
skyrmion_sides = skyrmion_mesh.boundary[2, 1] * skyrmion_edges  # [corners] Point-[faces]
skyrmion_patches = (skyrmion_sides[0] & skyrmion_sides[1]) / 2  # Patch-[faces]
skyrmion_areas = skyrmion_patches.norm()                   # Scalar+[faces]
skyrmion_gradients = (skyrmion_sides | skyrmion_patches.inverse()) / 2  # [corners] Plane+[faces]

face_rotors = skyrmion_mesh.incidence[2, 0] * skyrmion_rotors  # [corners] Rotor+[faces]
face_chords = face_rotors.sum(axis=0) / 3                  # Even+[faces]
face_differential = (face_rotors * (skyrmion_gradients & Point)).sum(axis=0)  # (Even+ <- Point-)[faces]
face_darboux = 2 * face_chords.reverse() * face_differential  # (Even+ <- Point-)[faces]
face_omega = face_darboux.select[2] / face_chords.scalar_norm_squared()  # (Bivector+ <- Point-)[faces]

# The cotangent weights turn edge readings into the direction-independent frame energy.
skyrmion_dots = -skyrmion_sides[[1, 2, 0]].dot(skyrmion_sides[[2, 0, 1]])  # [corners] Scalar+[faces]
skyrmion_cotangents = skyrmion_dots / (2 * skyrmion_areas) # [corners] Scalar+[faces]
skyrmion_energy = ((skyrmion_cotangents / (2 * skyrmion_areas)) * face_omega(skyrmion_sides).scalar_norm_squared()).sum(axis=0)  # Scalar+[faces]

# Reading the two turning rates against the director's equatorial patch gives its swept area.
# The full frame connection is flat: its derivative cancels this commutator term.
swept_turns = face_omega(skyrmion_sides[1]).commutator(face_omega(skyrmion_sides[0])) / 2  # Bivector+[faces]
solid_angles = swept_turns & mv.xy.dual()          # Scalar+[faces]
winding_density = solid_angles / skyrmion_areas          # Scalar+[faces]
skyrmion_charge = solid_angles.sites.sum() / (4 * np.pi)   # [] Scalar+

save_figure(
    render.draw_skyrmion(skyrmion_mesh, skyrmion_vertices, directors),
    "rotor_whitney_skyrmion",
);

Matched colours identify the same path on the disk and its image on the sphere. Radial paths become meridians; concentric rings become latitude circles and contract again at the south pole. The face readings `solid_angles` sample the area swept by this director map. Their sum approaches one sphere's area as the mesh is refined. The full frame connection is flat: its derivative cancels its commutator term, even while the projected director sweeps area.

In quaternion Maurer–Cartan notation, the full frame form $A=\bar q\,dq$ satisfies $dA+A\wedge A=0$. In the vector notation of the sigma model, the director winding reads $Q=\frac{1}{4\pi}\int\mathbf n\cdot(\partial_x\mathbf n\times\partial_y\mathbf n)\,dx\,dy$.


In [ ]:
print("Integrated Skyrmion topological charge Q:", skyrmion_charge.to_array())
print("Total frame Dirichlet energy:", (skyrmion_areas * skyrmion_energy).sites.sum().to_array())

## Extending boundary frames across a saddle

Boundary frames can guide a smooth orientation field across the whole surface. The linear harmonic chord field minimizes edge energy in the even subalgebra; normalizing gives a unit frame at each vertex. Different boundary twists produce visibly different ribbon patterns across the same saddle.


In [ ]:
n_saddle_divisions = 6
saddle_height = 0.28
relaxation_step_size = 0.18
n_relaxation_steps = 20
frame_duration_ms = 90

In [ ]:
saddle_mesh, saddle_vertices, boundary, prescribed_rotors, initial_rotors = scenarios.saddle_surface(
    disk_radius, n_saddle_divisions, saddle_height,
)
saddle_edges = saddle_mesh.d[0] * saddle_vertices          # Point-[edges]
saddle_sides = saddle_mesh.boundary[2, 1] * saddle_edges   # [corners] Point-[faces]
saddle_patches = (saddle_sides[0] & saddle_sides[1]) / 2   # Patch-[faces]
saddle_areas = saddle_patches.norm()                       # Scalar+[faces]
saddle_dots = -saddle_sides[[1, 2, 0]].dot(saddle_sides[[2, 0, 1]])  # [corners] Scalar+[faces]
saddle_cotangents = saddle_dots / (2 * saddle_areas)       # [corners] Scalar+[faces]
edge_weights = (~saddle_mesh.incidence[2, 1] * (saddle_cotangents / 2)).sum(axis=0)  # Scalar+[edges]

rotor_d0 = saddle_mesh.d[0] * Rotor                        # (Even+ <- Even+)[edges <- vertices]
rotor_laplacian = rotor_d0.adjoint()(spdiag(edge_weights) * rotor_d0)  # (Even+ <- Even+)[vertices <- vertices]
free_vertices = np.setdiff1d(np.arange(saddle_mesh.n_vertices), boundary)
free = ~SparseExtensor.selection(mv.context, free_vertices, saddle_mesh.n_vertices) * Rotor     # (Even+ <- Even+)[vertices <- free]
pinned = ~SparseExtensor.selection(mv.context, boundary, saddle_mesh.n_vertices) * Rotor        # (Even+ <- Even+)[vertices <- boundary]

# The chord field minimizes edge energy in the even subalgebra.
# Normalization gives unit frames, changing the variational problem in the process.
boundary_values = pinned(pinned.adjoint()(prescribed_rotors))  # [cases] Even+[vertices]
rhs = -free.adjoint()(rotor_laplacian(boundary_values))    # [cases] Even+[free]
harmonic_chords = boundary_values + free(free.adjoint()(rotor_laplacian(free)).solve(rhs))  # [cases] Even+[vertices]
harmonic_rotors = harmonic_chords.normalized()             # [cases] Rotor+[vertices]

save_figure(
    render.draw_rotor_frames(saddle_mesh, saddle_vertices, harmonic_rotors),
    "rotor_whitney_frames",
);

The static panels show gentle boundary tilt and two opposite boundary twists. The ribbons follow fixed paths on the surface; their cross-sections turn with two frame directions, making orientation visible as shape. In the animation an extra interior twist unwinds while the boundary stays pinned.

This relaxation keeps every rotor unit throughout, so it solves a different variational problem from normalizing a linear harmonic chord field. Its limiting field need not be the static interpolation above.

In the harmonic-map notation of Riemannian geometry, constrained smoothing minimizes $E(q)=\frac12\sum_{(i,j)}w_{ij}|q_j-q_i|^2$ subject to $q_i\in S^3$, with stationary equation $(Lq)_i=\langle q_i,(Lq)_i\rangle q_i$ at interior vertices.


In [ ]:
def relax_frames(start_rotors: Rotor, step_size: float, n_steps: int) -> Iterator[tuple[Rotor, Scalar]]:
    """Projected cotangent steps smooth the interior while preserving unit rotors and pinned boundary frames."""
    current = start_rotors                                 # Rotor+[vertices]
    for _ in range(n_steps + 1):
        differences = rotor_d0(current)                    # Even+[edges]
        energy = (edge_weights * differences.scalar_norm_squared()).sites.sum() / 2  # [] Scalar+
        yield current, energy
        current = (current - step_size * free(free.adjoint()(rotor_laplacian(current)))).normalized()  # Rotor+[vertices]

relaxation_states = relax_frames(initial_rotors, relaxation_step_size, n_relaxation_steps)
show_animation(
    render.animate_relaxation(saddle_mesh, saddle_vertices, relaxation_states),
    "rotor_whitney_relaxation",
    frame_duration_ms,
    n_frames=n_relaxation_steps + 1,
    forward_then_back=True,
)

In [ ]:
relaxation_energies = stack([
    energy for _, energy in relax_frames(initial_rotors, relaxation_step_size, n_relaxation_steps)
])                                                         # [steps + 1] Scalar+
print("Initial vs relaxed Dirichlet energy:", relaxation_energies[[0, -1]].to_array())
print("Normalized chord-field energies across the boundary cases:",
      ((edge_weights * rotor_d0(harmonic_rotors).scalar_norm_squared()).sites.sum() / 2).to_array())


In [ ]:
# checks
np.testing.assert_allclose((darboux.select[2] - whitney_rates).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((darboux.select[0] - chord_stretch).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose(unit_rotors.scalar_norm_squared().kernel, 1.0, atol=1e-14)

radial_whitney = ((points[:, None] - corners) | patch.inverse()) / 2
np.testing.assert_allclose((whitney_forms - radial_whitney).kernel, 0.0, atol=1e-14)

np.testing.assert_allclose(skyrmion_charge.kernel, 1.0, rtol=1.5e-2)
np.testing.assert_allclose(
    (scalar_parts.squared() + bivector_norms.squared()).kernel, 1.0, atol=1e-14,
)

np.testing.assert_allclose(free.adjoint()(rotor_laplacian(harmonic_chords)).kernel, 0.0, atol=1e-11)
assert np.all(relaxation_energies[1:] < relaxation_energies[:-1])